In [2]:
# importing libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from catboost import CatBoostClassifier

train_df = pd.read_pickle("../data/train.pkl")

In [3]:
from sklearn.metrics import (
    accuracy_score ,
    precision_score ,
    recall_score ,
    f1_score ,
    average_precision_score , 
    roc_auc_score ,
    confusion_matrix ,
    classification_report
)

In [4]:
train_df = train_df.sort_values(["Transaction_Day" , "Transaction_Time_Of_Day"]).reset_index(drop=True)

In [5]:
x = train_df.drop(columns=["isFraud" , "TransactionID" ,"TransactionDT"])
y = train_df["isFraud"]

In [6]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5 , gap=0)

In [ ]:
oof_predictions = np.full(len(x) , np.nan)

for fold , (train_idx , val_idx) in enumerate(tscv.split(x) , start=1):
    x_train = x.iloc[train_idx].copy()
    x_val = x.iloc[val_idx].copy()

    y_train = y.iloc[train_idx].copy()
    y_val = y.iloc[val_idx].copy()

    numeric_cols = x_train.select_dtypes(include=["int64" , "float64" , "uint16" , "uint8"]).columns
    categorical_cols = x_train.select_dtypes(include=["object"]).columns

    x_train[categorical_cols] = x_train[categorical_cols].fillna("Missing")
    x_val[categorical_cols] = x_val[categorical_cols].fillna("Missing")

    catboost_model = CatBoostClassifier(iterations=300 , learning_rate=0.05 , depth=6 , loss_function="Logloss" , eval_metric="AUC",
                                random_seed=42 , verbose=100)
    
    catboost_model.fit(x_train , y_train , cat_features = categorical_cols.tolist())
    val_pred = catboost_model.predict(x_val)
    val_prob = catboost_model.predict_proba(x_val)[:,1]
    oof_predictions[val_idx] = val_prob

    roc_auc = roc_auc_score(y_val , val_prob)
    pr_auc = average_precision_score(y_val , val_prob)
    recall = recall_score(y_val , val_pred)
    f1 = f1_score(y_val , val_pred)
    precision = precision_score(y_val , val_pred)


    print(
        f"Fold {fold} | "
        f"ROC-AUC : {roc_auc:.4f} | " 
        f"PR-AUC : {pr_auc:.4f} | "
        f"Recall = {recall:.4f} | "
        f"F1 Score = {f1:.4f} | "
        f"Precision = {precision:.4f}")
    print(confusion_matrix(y_val , val_pred))

oof_mask = ~np.isnan(oof_predictions)
overall_roc_auc = roc_auc_score(y.iloc[oof_mask] , oof_predictions[oof_mask])
overall_pr_auc = average_precision_score(y.iloc[oof_mask] , oof_predictions[oof_mask])
print(f"OOF ROC-AUC : {overall_roc_auc:.4f}")
print(f"OOF PR-AUC : {overall_pr_auc:.4f}")

0:	total: 1.53s	remaining: 7m 38s


In [ ]:
# Fold 1 | ROC-AUC : 0.8930 | PR-AUC : 0.5554 | Recall = 0.3287 | F1 Score = 0.4802 | Precision = 0.8910
# [[94983   133]
#  [ 2220  1087]]

# Fold 2 | ROC-AUC : 0.9151 | PR-AUC : 0.6193 | Recall = 0.4207 | F1 Score = 0.5666 | Precision = 0.8678
# [[94105   260]
#  [ 2351  1707]]

# Fold 3 | ROC-AUC : 0.9029 | PR-AUC : 0.5601 | Recall = 0.3538 | F1 Score = 0.5032 | Precision = 0.8711
# [[94544   193]
#  [ 2382  1304]]

# Fold 4 | ROC-AUC : 0.9268 | PR-AUC : 0.6306 | Recall = 0.4357 | F1 Score = 0.5764 | Precision = 0.8511
# [[94456   281]
#  [ 2080  1606]]

# Fold 5 | ROC-AUC : 0.9225 | PR-AUC : 0.6080 | Recall = 0.4189 | F1 Score = 0.5542 | Precision = 0.8188
# [[94711   315]
#  [ 1974  1423]]

In [ ]:
catboost_oof = pd.DataFrame({
    "isFraud" : y.values ,
    "catboost_oof" : oof_predictions
})

catboost_oof.to_csv("../outputs/oof/catboost_oof.csv" , index=False)

In [ ]:
print(f"Shape : {catboost_oof.shape}")
print(f"Valid OOF predictions : {catboost_oof['catboost_oof'].notna().sum()}")
print(f"Missing OOF predictions : {catboost_oof['catboost_oof'].isna().sum()}")

Shape : (590540, 2)
Valid OOF predictions : 492115
Missing OOF predictions : 98425
